# Логистички модел и ML pipeline за население

**Семинарска работа** — свет vs. Македонија (World Bank, 1960–2024)

Самостоен notebook — целата анализа е во овие клетки, чекор по чекор:
1. Преземање и претобработка на податоци
2. Логистички модел (ODE)
3. ML pipeline за краткорочно предвидување
4. Визуелизации и зачувување на резултати


## Поставување

Потребни библиотеки: `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`, `scikit-learn`.

Следната клетка ги дефинира константите и помошните функции (податоци, логистички модел, ML, графици).


In [ ]:
%matplotlib inline

import json
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from scipy.optimize import curve_fit
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
FIG_DIR = BASE_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10
sns.set_style("whitegrid")

WB_API = "https://api.worldbank.org/v2/country/{code}/indicator/SP.POP.TOTL"
FORECAST_YEARS = 10
MK_GROWTH_END = 1990
TRAIN_END_YEAR = 2010
SELECT_K = 6


# ============================================================
# 1. Податоци
# ============================================================
def fetch_population(country_code, label):
    """Преземи податоци од World Bank API (SP.POP.TOTL, 1960–2024)."""
    path = DATA_DIR / f"{label}_population.csv"
    if path.exists():
        return pd.read_csv(path)

    url = (
        f"{WB_API.format(code=country_code)}"
        f"?format=json&per_page=100&date=1960:2024"
    )
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = json.load(response)

    rows = [
        (int(item["date"]), item["value"])
        for item in payload[1]
        if item["value"] is not None
    ]
    df = pd.DataFrame(rows, columns=["year", "population"]).sort_values("year")
    df.to_csv(path, index=False)
    return df


def preprocess_population(df):
    """Претобработка: сортирање, проверка на празнини, лог-трансформација (опционална колона)."""
    clean = df.sort_values("year").reset_index(drop=True).copy()
    clean["population"] = clean["population"].astype(float)
    clean["log_population"] = np.log(clean["population"])
    missing = clean["population"].isna().sum()
    if missing:
        clean["population"] = clean["population"].interpolate(method="linear")
        clean["log_population"] = np.log(clean["population"])
    return clean


# ============================================================
# 2. Логистички модел (примена математика)
# ============================================================
def logistic(t, K, r, P0):
    return K / (1.0 + ((K - P0) / P0) * np.exp(-r * t))


def logistic_derivative(t, K, r, P0):
    p = logistic(t, K, r, P0)
    return r * p * (1.0 - p / K)


def fit_logistic(df, end_year=None):
    subset = df if end_year is None else df[df["year"] <= end_year].copy()
    year0 = int(subset["year"].iloc[0])
    P0 = float(subset["population"].iloc[0])
    t = subset["year"].values - year0
    y = subset["population"].values.astype(float)

    popt, _ = curve_fit(
        lambda tt, K, r: logistic(tt, K, r, P0),
        t,
        y,
        p0=[y.max() * 1.5, 0.02],
        bounds=([y.max() * 0.9, 1e-4], [y.max() * 25, 0.25]),
        maxfev=30000,
    )
    K, r = popt
    y_hat = logistic(t, K, r, P0)

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    rmse = float(np.sqrt(np.mean((y - y_hat) ** 2)))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    mape = float(np.mean(np.abs((y - y_hat) / y)) * 100)

    last_year = int(subset["year"].max())
    future_years = np.arange(last_year + 1, last_year + FORECAST_YEARS + 1)
    t_future = future_years - year0
    future_pop = logistic(t_future, K, r, P0)

    return {
        "K": float(K),
        "r": float(r),
        "P0": P0,
        "year0": year0,
        "end_year": int(subset["year"].max()),
        "rmse": rmse,
        "r2": r2,
        "mape": mape,
        "doubling_time": float(np.log(2) / r),
        "t": t,
        "years": subset["year"].values,
        "observed": y,
        "fitted": y_hat,
        "future_years": future_years.tolist(),
        "future_pop": future_pop.tolist(),
    }


def plot_series(ax, df, fit, title, color, unit="millions"):
    """unit: 'billions' за свет, 'millions' за Македонија."""
    if unit == "billions":
        scale = 1e9
        unit_label = "милијарди"
    else:
        scale = 1e6
        unit_label = "милиони"
    t_all = df["year"].values - fit["year0"]
    curve = logistic(t_all, fit["K"], fit["r"], fit["P0"])

    ax.scatter(
        df["year"],
        df["population"] / scale,
        s=28,
        color=color,
        edgecolors="white",
        linewidths=0.4,
        label="Набљудувани податоци",
        zorder=3,
    )
    ax.plot(df["year"], curve / scale, color=color, linewidth=2, label="Логистички модел")
    ax.plot(
        fit["future_years"],
        np.array(fit["future_pop"]) / scale,
        color=color,
        linestyle="--",
        linewidth=2,
        label=f"Прогноза ({FORECAST_YEARS} г.)",
    )
    ax.axvline(fit["end_year"], color="gray", linestyle=":", alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel("Година")
    ax.set_ylabel(f"Население ({unit_label})")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)


# ============================================================
# 3. ML pipeline
# ============================================================
def build_features(df):
    """Екстракција на карактеристики од временска серија (без leakage)."""
    data = df.sort_values("year").copy()
    data["years_since_start"] = data["year"] - data["year"].iloc[0]
    data["year_squared"] = data["years_since_start"] ** 2
    data["pop_lag1"] = data["population"].shift(1)
    data["pop_lag2"] = data["population"].shift(2)
    data["pop_lag3"] = data["population"].shift(3)
    data["growth_rate"] = data["population"].pct_change()
    data["growth_rate_lag1"] = data["growth_rate"].shift(1)
    data["rolling_mean_3"] = data["population"].shift(1).rolling(3).mean()
    data["rolling_mean_5"] = data["population"].shift(1).rolling(5).mean()
    data["rolling_std_5"] = data["population"].shift(1).rolling(5).std()
    data["decade"] = (data["year"] // 10) * 10

    feature_cols = [
        "years_since_start",
        "year_squared",
        "pop_lag1",
        "pop_lag2",
        "pop_lag3",
        "growth_rate_lag1",
        "rolling_mean_3",
        "rolling_mean_5",
        "rolling_std_5",
    ]
    data = data.dropna().reset_index(drop=True)
    return data, feature_cols


def regression_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    mape = float(mean_absolute_percentage_error(y_true, y_pred) * 100)
    return {"RMSE": rmse, "R2": r2, "MAPE": mape}


def run_ml_pipeline(df, label, logistic_fit):
    """Целосен ML pipeline: features → selection → models → tuning."""
    print(f"\n=== ML PIPELINE: {label} ===")
    featured, feature_cols = build_features(df)

    train = featured[featured["year"] <= TRAIN_END_YEAR]
    test = featured[featured["year"] > TRAIN_END_YEAR]
    X_train = train[feature_cols]
    y_train = train["population"]
    X_test = test[feature_cols]
    y_test = test["population"]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Feature selection
    selector = SelectKBest(score_func=f_regression, k=SELECT_K)
    selector.fit(X_train_scaled, y_train)
    scores = pd.Series(selector.scores_, index=feature_cols).sort_values(ascending=False)
    selected_kbest = scores.head(SELECT_K).index.tolist()

    rfe = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=SELECT_K)
    rfe.fit(X_train_scaled, y_train)
    selected_rfe = [f for f, keep in zip(feature_cols, rfe.support_) if keep]

    top_features = selected_kbest
    feat_idx = [feature_cols.index(f) for f in top_features]
    X_train_sel = X_train_scaled[:, feat_idx]
    X_test_sel = X_test_scaled[:, feat_idx]

    print("SelectKBest:", selected_kbest)
    print("RFE:", selected_rfe)

    models = {
        "Linear Regression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "KNN": KNeighborsRegressor(n_neighbors=3),
    }

    tscv = TimeSeriesSplit(n_splits=5)
    results = []
    trained = {}

    for name, model in models.items():
        cv_scores = cross_val_score(
            model, X_train_sel, y_train, cv=tscv, scoring="neg_root_mean_squared_error"
        )
        model.fit(X_train_sel, y_train)
        y_pred = model.predict(X_test_sel)
        metrics = regression_metrics(y_test, y_pred)
        results.append(
            {
                "Region": label,
                "Model": name,
                "CV RMSE mean": float(-cv_scores.mean()),
                "CV RMSE std": float(cv_scores.std()),
                **metrics,
            }
        )
        trained[name] = model

    # Логистички baseline на тест период
    t_test = test["year"].values - logistic_fit["year0"]
    logistic_pred = logistic(t_test, logistic_fit["K"], logistic_fit["r"], logistic_fit["P0"])
    log_metrics = regression_metrics(y_test, logistic_pred)
    results.append(
        {
            "Region": label,
            "Model": "Logistic (baseline)",
            "CV RMSE mean": np.nan,
            "CV RMSE std": np.nan,
            **log_metrics,
        }
    )

    results_df = pd.DataFrame(results).sort_values("RMSE")

    # Hyperparameter tuning — Gradient Boosting
    param_grid = {
        "n_estimators": [50, 100, 150],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.05, 0.1, 0.2],
    }
    grid = GridSearchCV(
        GradientBoostingRegressor(random_state=42),
        param_grid,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    grid.fit(X_train_sel, y_train)
    tuned_pred = grid.best_estimator_.predict(X_test_sel)
    tuned_metrics = regression_metrics(y_test, tuned_pred)
    print(f"GridSearch best: {grid.best_params_}, test RMSE={tuned_metrics['RMSE']:,.0f}")

    return {
        "featured": featured,
        "feature_cols": feature_cols,
        "top_features": top_features,
        "scores": scores,
        "train": train,
        "test": test,
        "results_df": results_df,
        "trained": trained,
        "logistic_pred": logistic_pred,
        "grid_search": {
            "best_params": grid.best_params_,
            "best_cv_rmse": float(-grid.best_score_),
            "test_metrics": tuned_metrics,
        },
        "tuned_model": grid.best_estimator_,
        "tuned_pred": tuned_pred,
        "X_train_sel": X_train_sel,
        "y_train": y_train,
        "X_test_sel": X_test_sel,
        "y_test": y_test,
        "cv_results": pd.DataFrame(grid.cv_results_),
    }


def plot_logistic_figures(world_df, mk_df, world_fit, mk_fit, mk_growth_fit):
    fig, ax = plt.subplots(figsize=(9, 5))
    plot_series(ax, world_df, world_fit, "Светско население и логистички модел", "#1f77b4", unit="billions")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig1_world_logistic.png"), bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 5))
    plot_series(ax, mk_df, mk_fit, "Население на Северна Македонија (цел период)", "#d62728")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig2_macedonia_logistic.png"), bbox_inches="tight")
    plt.close()

    mk_growth_df = mk_df[mk_df["year"] <= MK_GROWTH_END]
    fig, ax = plt.subplots(figsize=(9, 5))
    plot_series(
        ax, mk_growth_df, mk_growth_fit,
        f"Македонија — фаза на раст (1960–{MK_GROWTH_END})", "#2ca02c",
    )
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig3_macedonia_growth_phase.png"), bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 5))
    for df, fit, lbl, color in [
        (world_df, world_fit, "Свет", "#1f77b4"),
        (mk_df, mk_fit, "Македонија", "#d62728"),
    ]:
        t = df["year"].values - fit["year0"]
        ax.plot(df["year"], logistic(t, fit["K"], fit["r"], fit["P0"]) / fit["K"], linewidth=2, color=color, label=lbl)
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5, label="Капацитет K")
    ax.set_title("Нормализиран раст: P(t) / K")
    ax.set_xlabel("Година")
    ax.set_ylabel("P / K")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig4_normalized_comparison.png"), bbox_inches="tight")
    plt.close()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, df, fit, title, res_scale, res_unit in [
        (axes[0], world_df, world_fit, "Свет — резидуали", 1e6, "милиони"),
        (axes[1], mk_df, mk_fit, "Македонија — резидуали", 1e3, "илјади"),
    ]:
        t = df["year"].values - fit["year0"]
        residuals = df["population"].values - logistic(t, fit["K"], fit["r"], fit["P0"])
        ax.bar(df["year"], residuals / res_scale, color="steelblue", width=0.8)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("Година")
        ax.set_ylabel(f"Грешка ({res_unit})")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig5_residuals.png"), bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 5))
    for df, fit, lbl, color in [
        (world_df, world_fit, "Свет (емпириски)", "#1f77b4"),
        (mk_df, mk_fit, "Македонија (емпириски)", "#d62728"),
    ]:
        years = df["year"].values
        pop = df["population"].values
        empirical_rate = np.diff(pop) / pop[:-1] / np.diff(years)
        ax.plot(years[1:], empirical_rate * 100, "o-", ms=3, label=lbl, color=color)

    t_dense = np.linspace(0, world_df["year"].max() - world_fit["year0"], 200)
    p_dense = logistic(t_dense, world_fit["K"], world_fit["r"], world_fit["P0"])
    rate_dense = logistic_derivative(t_dense, world_fit["K"], world_fit["r"], world_fit["P0"]) / p_dense * 100
    ax.plot(t_dense + world_fit["year0"], rate_dense, color="#1f77b4", linestyle="--", linewidth=2, label="Свет (модел)")
    ax.set_title("Годишна релативна стапка на раст (%)")
    ax.set_xlabel("Година")
    ax.set_ylabel("(ΔP/P) / Δt  [%]")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig6_growth_rates.png"), bbox_inches="tight")
    plt.close()


def plot_ml_figures(world_ml, mk_ml):
    # Fig 7: EDA correlation heatmap (world features)
    corr_cols = world_ml["top_features"] + ["population"]
    corr = world_ml["featured"][corr_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
    ax.set_title("EDA — корелациска матрица (свет, ML карактеристики)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig7_feature_correlation.png"), bbox_inches="tight")
    plt.close()

    # Fig 8: Feature scores
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, ml, title in [
        (axes[0], world_ml, "Свет"),
        (axes[1], mk_ml, "Македонија"),
    ]:
        ml["scores"].sort_values().plot(kind="barh", ax=ax, color="teal")
        ax.set_title(f"SelectKBest — {title}")
        ax.set_xlabel("F-score")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig8_feature_scores.png"), bbox_inches="tight")
    plt.close()

    # Fig 9: Model comparison
    combined = pd.concat([world_ml["results_df"], mk_ml["results_df"]], ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, region, ml, rmse_scale, rmse_unit in [
        (axes[0], "свет", world_ml, 1e6, "милиони"),
        (axes[1], "Македонија", mk_ml, 1.0, "лица"),
    ]:
        sub = ml["results_df"].sort_values("RMSE")
        x = np.arange(len(sub))
        ax.bar(x, sub["RMSE"] / rmse_scale, color="steelblue", edgecolor="black")
        ax.set_xticks(x)
        ax.set_xticklabels(sub["Model"], rotation=25, ha="right", fontsize=8)
        ax.set_title(f"Споредба на модели — {region}")
        ax.set_ylabel(f"RMSE ({rmse_unit})")
        ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig9_model_comparison.png"), bbox_inches="tight")
    plt.close()
    combined.to_csv(BASE_DIR / "model_results.csv", index=False)

    # Fig 10: Actual vs predicted (best ML + logistic) on test
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, ml, title, scale, unit_label in [
        (axes[0], world_ml, "Свет", 1e9, "милијарди"),
        (axes[1], mk_ml, "Македонија", 1e6, "милиони"),
    ]:
        test = ml["test"]
        best_name = ml["results_df"].iloc[0]["Model"]
        if best_name == "Logistic (baseline)":
            best_pred = ml["logistic_pred"]
        else:
            best_pred = ml["trained"][best_name].predict(ml["X_test_sel"])
        ax.plot(test["year"], test["population"] / scale, "o-", label="Вистинско", color="black")
        ax.plot(test["year"], best_pred / scale, "s--", label=f"Најдобар ML: {best_name}")
        ax.plot(test["year"], ml["tuned_pred"] / scale, "^--", label="Tuned GB", alpha=0.8)
        ax.plot(test["year"], ml["logistic_pred"] / scale, "x:", label="Логистички")
        ax.set_title(title)
        ax.set_xlabel("Година")
        ax.set_ylabel(f"Население ({unit_label})")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig10_test_predictions.png"), bbox_inches="tight")
    plt.close()

    # Fig 11: GridSearch heatmap (world GB)
    cv = world_ml["cv_results"]
    pivot = cv.groupby(["param_max_depth", "param_n_estimators"])["mean_test_score"].max().unstack()
    pivot = -pivot  # back to positive RMSE
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGn_r", ax=ax)
    ax.set_title("GridSearchCV — Gradient Boosting (свет)")
    ax.set_xlabel("n_estimators")
    ax.set_ylabel("max_depth")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig11_gridsearch_heatmap.png"), bbox_inches="tight")
    plt.close()


FIG_PATH = FIG_DIR
print(f"Проект: {BASE_DIR}")
print(f"Слики: {FIG_PATH}")


## 1. Податоци

Податоците се преземаат од **World Bank Open Data** (индикатор `SP.POP.TOTL`).
Ако CSV веќе постои во `data/`, се вчитува локално — инаку се симнува преку API.


In [ ]:
world_df = preprocess_population(fetch_population("WLD", "world"))
mk_df = preprocess_population(fetch_population("MKD", "macedonia"))

print(f"Свет: {len(world_df)} години ({world_df.year.min()}–{world_df.year.max()})")
print(f"Македонија: {len(mk_df)} години ({mk_df.year.min()}–{mk_df.year.max()})")

display(world_df.tail(3))
display(mk_df.tail(3))


**Население во 2024:** свет ≈ 8,14 милијарди; Македонија ≈ 1,82 милиони.
Македонија го достигнува максимумот во **1990** (≈ 2,06 милиони), потоа опаѓа.


In [ ]:
pop_2024_world = world_df.loc[world_df.year == 2024, "population"].iloc[0]
pop_2024_mk = mk_df.loc[mk_df.year == 2024, "population"].iloc[0]
peak_mk = mk_df.loc[mk_df["population"].idxmax()]

print(f"Свет 2024: {pop_2024_world/1e9:.2f} милијарди")
print(f"МК 2024: {pop_2024_mk/1e6:.2f} милиони")
print(f"МК пик {int(peak_mk.year)}: {peak_mk.population/1e6:.2f} милиони")


## 2. Логистички модел

Моделот опишува раст со равенка:

**dP/dt = r · P · (1 − P/K)**

каде **K** е носечки капацитет, **r** стапка на раст. Параметрите се проценети со `scipy.optimize.curve_fit`.


In [ ]:
world_fit = fit_logistic(world_df)
mk_fit = fit_logistic(mk_df)
mk_growth_fit = fit_logistic(mk_df, end_year=MK_GROWTH_END)

for name, fit in [
    ("Свет", world_fit),
    ("Македонија (цел период)", mk_fit),
    (f"Македонија (1960–{MK_GROWTH_END})", mk_growth_fit),
]:
    print(f"{name}: K={fit['K']:,.0f}, r={fit['r']:.5f}, R²={fit['r2']:.4f}")


**Интерпретација:**
- **Свет** — R² ≈ 0,9999: одлично прилагодување
- **МК цел период** — R² ≈ 0,82: моделот не го фаќа опаѓањето по 1990
- **МК 1960–1990** — R² ≈ 0,991: добро за фазата на раст

Слики 1–6: трендови, нормализиран P/K, резидуали, стапки на раст.


In [ ]:
plot_logistic_figures(world_df, mk_df, world_fit, mk_fit, mk_growth_fit)

logistic_figs = [
    "fig1_world_logistic", "fig2_macedonia_logistic", "fig3_macedonia_growth_phase",
    "fig4_normalized_comparison", "fig5_residuals", "fig6_growth_rates",
]
for name in logistic_figs:
    path = FIG_PATH / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path), width=700))


## 3. ML pipeline

За **краткорочно предвидување** се гради ML pipeline:
- **Train:** 1965–2010 | **Test:** 2011–2024
- 9 карактеристики (лаг, rolling просеци, стапка на раст…)
- SelectKBest + RFE → 6 карактеристики
- 5 регресиски модели + логистички baseline
- GridSearchCV на Gradient Boosting


In [ ]:
world_ml = run_ml_pipeline(world_df, "world", world_fit)
mk_ml = run_ml_pipeline(mk_df, "macedonia", mk_fit)

print("Топ карактеристики (свет):", world_ml["top_features"])
print("Топ карактеристики (МК):", mk_ml["top_features"])


## 4. ML резултати (test 2011–2024)

Споредба по **RMSE**, **R²** и **MAPE**. Очекувано: Linear Regression најдобар; ensemble моделите overfit-ираат.


In [ ]:
print("=== СВЕТ ===")
display(world_ml["results_df"].round(4))

print("=== МАКЕДОНИЈА ===")
display(mk_ml["results_df"].round(4))


Слики 7–11: корелациска матрица, F-scores, споредба на модели, предвидувања, GridSearchCV.


In [ ]:
plot_ml_figures(world_ml, mk_ml)

ml_figs = [
    "fig7_feature_correlation", "fig8_feature_scores", "fig9_model_comparison",
    "fig10_test_predictions", "fig11_gridsearch_heatmap",
]
for name in ml_figs:
    path = FIG_PATH / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path), width=700))


## 5. Зачувување на резултати

Резултатите се запишуваат во `results_summary.json` и `model_results.csv`.


In [ ]:
summary = {
    "data_source": "World Bank Open Data, indicator SP.POP.TOTL",
    "ml_train_period": f"1965–{TRAIN_END_YEAR}",
    "ml_test_period": f"{TRAIN_END_YEAR + 1}–2024",
    "world": {
        "population_2024": int(pop_2024_world),
        "logistic_r2": world_fit["r2"],
        "best_ml_model": world_ml["results_df"].iloc[0]["Model"],
        "best_ml_rmse": float(world_ml["results_df"].iloc[0]["RMSE"]),
    },
    "macedonia": {
        "population_2024": int(pop_2024_mk),
        "peak_year": int(peak_mk.year),
        "logistic_r2_full": mk_fit["r2"],
        "logistic_r2_growth": mk_growth_fit["r2"],
        "best_ml_model": mk_ml["results_df"].iloc[0]["Model"],
        "best_ml_rmse": float(mk_ml["results_df"].iloc[0]["RMSE"]),
    },
}

out = BASE_DIR / "results_summary.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Зачувано: {out}")
print("=== АНАЛИЗАТА ЗАВРШЕНА ===")
